# 07 · Monitor: what retrieval picked for real questions

*For learning. Nothing here runs in CI and nothing leaves this computer: it reads the drill server's own records on this
computer (the progress database in `.drill/`, and the traces on the local MLflow server, `http://localhost:5050`). It
changes nothing and logs no run: it only looks.*

**What changed in the app.** Until now the coach never searched for anything: on a lesson page, the page sends that
lesson's text with the question. Some questions come with **no lesson**: a follow-up typed on a study list item, a
question through Claude's tools (MCP), any request without a lesson. For those the drill server now does **retrieval**
(finding the paragraphs that answer a question): it embeds the question with `bge-large`, the champion that stage 4
kept, finds the five closest Learn paragraphs, and puts them in front of the question in the coach's prompt. Putting
found text into the prompt is **retrieval-augmented generation (RAG)** (the course's *Retrieval: finding the right
chunks*; glossary: RAG). The code is `src/drill/server/retrieve.py`; `docs/SERVER.md` says what it records.

**What is recorded for each such question**, so it can be watched now and A/B tested later:

| where | what |
|---|---|
| the `asked` event (the progress database) | `retrieval` (`on`, `off`, `error` or `timeout`), `retrieved` (the lessons found, closest first), `retrieval_model`, `retrieval_ms`, and the question |
| the MLflow trace | a `retrieve` span of type RETRIEVER with the question searched and the paragraphs found, and a `retrieval` tag |

A **span** is one step inside a trace, with its own timing (glossary: trace, span). This notebook reads both, and says
plainly when there is nothing to read yet.

In [1]:
import json
import os
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import httpx
import pandas as pd
from sqlalchemy import create_engine, select

from drill import mlflow_names
from drill.server.bank import Bank
from drill.server.store import events, local_url

pd.set_option("display.max_colwidth", 70)
DB_URL = os.environ.get("DRILL_DATABASE_URL", "").strip() or local_url()   # the server's own default, unless you name another
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "").strip() or "http://localhost:5050"
EXPERIMENT = mlflow_names.experiment()   # where the drill server writes its traces
bank = Bank.load()
title = {lid: lesson["title"] for lid, lesson in bank.lessons.items()}
print(f"{len(bank.lessons)} lessons; reading events from {Path(DB_URL.removeprefix('sqlite:///')).name + ' in .drill/' if DB_URL.startswith('sqlite') else 'the database DRILL_DATABASE_URL names'}"
      f" and traces from experiment {EXPERIMENT!r}")

213 lessons; reading events from drill.db in .drill/ and traces from experiment 'daily-drill'


## 1. The questions that came with no lesson

Every `asked` event is one request to the coach. The ones with an empty lesson are the ones retrieval is for. Rows written
before retrieval existed have no `retrieval` field, so they are counted apart. Only this computer's own records are read:
at work, the same rows are in the managers' reports (`drill_insights.activity`, the columns `retrieval_ran` and
`retrieved_lesson`).

In [2]:
def asked_events(url: str) -> list[dict]:
    """Every 'asked' event: when, which lesson (empty for none), and its detail. Nothing is created if the file is missing."""
    if url.startswith("sqlite:///") and not Path(url.removeprefix("sqlite:///")).is_file():
        return []
    engine = create_engine(url)
    with engine.connect() as conn:
        rows = conn.execute(select(events.c.at, events.c.lesson_id, events.c.detail).where(events.c.kind == "asked").order_by(events.c.at)).all()
    engine.dispose()
    return [{"at": at, "lesson": lid or "", **json.loads(detail)} for at, lid, detail in rows]


asked = asked_events(DB_URL)
no_lesson = [a for a in asked if not a["lesson"]]
searched = [a for a in no_lesson if "retrieval" in a]
# Another database, such as a preview server's: DRILL_DATABASE_URL=sqlite:///.drill/preview.db before starting Jupyter.
print(f"{len(asked)} questions to the coach recorded here; {len(no_lesson)} came with no lesson; "
      f"{len(searched)} of those carry a retrieval record.")
if not searched:
    print("None to show yet (0 rows). The record starts with this change (2026-11-08). Ask a follow-up question on a "
          "study list item in the page, served by `uv run drill-server`, and it appears here.")

0 questions to the coach recorded here; 0 came with no lesson; 0 of those carry a retrieval record.
None to show yet (0 rows). The record starts with this change (2026-11-08). Ask a follow-up question on a study list item in the page, served by `uv run drill-server`, and it appears here.


In [3]:
if searched:
    table = pd.DataFrame([{
        "when (UTC)": datetime.fromtimestamp(a["at"], timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "for": a.get("feature", ""),
        "retrieval": a["retrieval"],
        "ms": a.get("retrieval_ms"),
        "closest lesson": title.get((a.get("retrieved") or [""])[0], (a.get("retrieved") or ["(none)"])[0]),
        "question": (a.get("question") or "")[:70],
        "why not": a.get("retrieval_why", ""),
    } for a in searched])
    states = Counter(a["retrieval"] for a in searched)
    print("How each search went:", dict(states))
    ran = [a["retrieval_ms"] for a in searched if a["retrieval"] == "on"]
    if ran:
        print(f"Milliseconds per search that ran: middle value {sorted(ran)[len(ran) // 2]}, slowest {max(ran)}")
    display(table)
else:
    print("No table: nothing has been searched on this computer yet.")

No table: nothing has been searched on this computer yet.


> **In plain words: what a timeout or an error means here.** The coach waits at most two seconds for the question to be
> embedded (`DRILL_RETRIEVAL_SECONDS`). Past that, or if the embedding model fails, it answers exactly as it did before
> this change, with no paragraphs added, and the record says `timeout` or `error` with the reason. Falling back to the old
> behaviour instead of failing is a **fallback** (glossary: fallback, timeout). `off` means it did not try: retrieval
> switched off, no question text (a study list quiz sends none), the paragraphs still being embedded at start-up, or no
> room in the prompt's size limit.

## 2. The same questions in the traces

The trace holds what the event cannot: the paragraphs themselves and how close each was (the similarity score, from -1
to 1; higher is closer). Traces live on the MLflow server, so this cell asks it first and says so if it is not running.

In [4]:
def retrieval_traces() -> list:
    try:
        httpx.get(TRACKING_URI.rstrip("/") + "/health", timeout=1.5).raise_for_status()
    except Exception as exc:
        print(f"The MLflow server at {TRACKING_URI} is not answering ({type(exc).__name__}), so no traces are read.")
        return []
    import mlflow

    mlflow.set_tracking_uri(TRACKING_URI)
    experiment = mlflow.get_experiment_by_name(EXPERIMENT)
    if experiment is None:
        print(f"No experiment {EXPERIMENT!r} on {TRACKING_URI} yet: the drill server has written no traces there.")
        return []
    found = []
    for state in ("on", "off", "error", "timeout"):
        found += mlflow.search_traces(locations=[experiment.experiment_id], filter_string=f"tags.retrieval = '{state}'",
                                      return_type="list", max_results=200)
    return [mlflow.get_trace(t.info.trace_id) for t in found]


traces = retrieval_traces()
rows = []
for trace in traces:
    span = next((s for s in trace.data.spans if s.span_type == "RETRIEVER"), None)
    if span is None:
        continue
    docs = span.outputs or []
    rows.append({
        "when (UTC)": datetime.fromtimestamp(trace.info.request_time / 1000, timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "for": trace.info.tags.get("feature", ""),
        "retrieval": span.attributes.get("retrieval"),
        "ms": span.attributes.get("retrieval_ms"),
        "closest paragraph": docs[0]["metadata"]["title"] if docs else "(none)",
        "score": docs[0]["metadata"]["score"] if docs else None,
        "fifth score": docs[-1]["metadata"]["score"] if len(docs) == 5 else None,
        "question": str((span.inputs or {}).get("query", ""))[:60],
    })
print(f"{len(rows)} {'trace' if len(rows) == 1 else 'traces'} with a retrieval span in {EXPERIMENT!r}.")
if rows:
    display(pd.DataFrame(rows).sort_values("when (UTC)"))
else:
    print("None to show yet (0 rows).")

1 trace with a retrieval span in 'daily-drill'.


,when (UTC),for,retrieval,ms,closest paragraph,score,fifth score,question
0,2026-11-09 02:17,ask_study_item,on,33,Chunking documents,0.7117,0.6603,why would I make the pieces overlap when I split a long docu


In [5]:
if rows:
    top = [r for r in rows if r["score"] is not None]
    print("The closest paragraph for each traced question, and the fifth:")
    for r in top:
        print(f"  {r['score']:.3f} … {r['fifth score'] if r['fifth score'] is not None else '-'}  {r['closest paragraph']}  ←  {r['question']}")
    print("A question whose closest score is far below the others' is one the course may not answer: worth reading.")
else:
    print("Nothing to compare yet.")

The closest paragraph for each traced question, and the fifth:
  0.712 … 0.6603  Chunking documents  ←  why would I make the pieces overlap when I split a long docu
A question whose closest score is far below the others' is one the course may not answer: worth reading.


## 3. What to watch once real questions arrive, and what an A/B test still needs

*Hunches, not findings: there are no real learner questions on this computer yet, so none of this has been measured.*

- **Which lessons people needed.** Count the closest lesson over a month (`retrieved_lesson` in `drill_insights.activity`
  at work). A lesson that keeps coming up for study list questions is one people go back to.
- **Whether it helps.** The measure the lab set for stage 7 is what learners do next: Got it against Still stuck, and
  whether the next Check question goes right (the README, "Online second"). A study list answer has no Got it or Still
  stuck rating of its own today (the item's Got it button says the learner understands the item, and is kept with the
  item, not on the answer's trace), so that signal has to be joined in, or added, before it can be read.
- **Drift.** If the closest scores fall over the weeks while the lessons stay the same, people are asking about things the
  course does not cover, or in new words (the course's *Drift monitoring*; glossary: data drift).
- **An A/B test** (glossary: A/B test) splits real questions between the champion and a challenger and compares what
  happens next. What is recorded now is enough for the champion's side: the model's name is on every record
  (`retrieval_model`). Still to build: a rule that sends a fixed share of people to the challenger, and a record of which
  side each question went to. There is no challenger to send them to yet: stage 4 kept the champion.